# Mini Proyecto: Diseño de una IA Sencilla en Python
### Cloud, IA y Data Science — Ciclo 1

---

En esta sesión vamos a construir un **sistema de clasificación automática** de principio a fin. No se trata de ver conceptos sueltos, sino de ensamblarlos en un flujo de trabajo real, el mismo que usaría un Data Scientist en su día a día.

El proyecto tiene seis fases:

1. **Definición del problema** — ¿Qué queremos predecir y por qué?
2. **Carga y exploración de datos** — ¿Con qué datos trabajamos?
3. **Preprocesamiento** — ¿Cómo dejamos los datos listos para el modelo?
4. **Entrenamiento del modelo** — ¿Cómo aprende la máquina?
5. **Evaluación** — ¿Cómo sabemos si el modelo es bueno?
6. **Reflexión y próximos pasos** — ¿Qué mejoraríamos?

> **Importante:** La sesión anterior trabajamos con las herramientas del ecosistema de IA (scikit-learn, etc.). Hoy las vamos a usar juntas, en contexto. Si algo te suena de antes, bien; si no, el código está explicado celda a celda.

---
## Fase 1 — Definición del problema

Antes de escribir una sola línea de código, un buen proyecto de Machine Learning comienza con una pregunta clara.

**Nuestro problema:** Un jardín botánico quiere automatizar la clasificación de tres especies de flor del género *Iris* a partir de mediciones físicas de sus pétalos y sépalos. Actualmente lo hace un experto a mano; queremos que un modelo lo haga de forma automática.

**¿Qué tipo de problema es este?**

Es un problema de **clasificación supervisada**: tenemos ejemplos ya etiquetados (flores cuya especie conocemos) y queremos que el modelo aprenda a etiquetar flores nuevas.

**¿Qué datos tenemos?**

El dataset Iris contiene 150 flores, cada una con cuatro características numéricas:
- Longitud del sépalo (cm)
- Anchura del sépalo (cm)
- Longitud del pétalo (cm)
- Anchura del pétalo (cm)

Y una etiqueta de clase: *Setosa*, *Versicolor* o *Virginica*.

> **Reflexión antes de continuar:** ¿Por qué es importante definir el problema antes de elegir el algoritmo? Muchos proyectos fracasan porque se empieza por el modelo en lugar de por la pregunta.

---
## Fase 2 — Carga y exploración de datos

### 2.1 Importar las librerías necesarias

Lo primero es importar todo lo que vamos a necesitar. Es una buena práctica hacerlo al principio del notebook para que cualquier lector sepa de un vistazo de qué depende el proyecto.

Usamos:
- `pandas` para manipular datos en forma de tabla
- `matplotlib` y `seaborn` para visualizaciones
- `sklearn` (scikit-learn) para cargar el dataset, preprocesar y modelar

> **Posible error frecuente:** Si alguna librería no está instalada, Python lanzará un `ModuleNotFoundError`. La solución siempre es `pip install nombre_librería` en el terminal antes de abrir el notebook.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

print("✅ Librerías importadas correctamente")

### 2.2 Cargar el dataset

scikit-learn incluye varios datasets de práctica que podemos cargar directamente, sin descargar ningún archivo. El dataset Iris es uno de ellos.

Lo vamos a convertir a un `DataFrame` de pandas para poder explorarlo con las herramientas que ya conocemos. Este paso de convertir el objeto de sklearn a DataFrame es muy habitual en la práctica.

In [ ]:
# Cargamos el dataset desde scikit-learn
iris = load_iris()

# Convertimos a DataFrame para trabajar más cómodamente
df = pd.DataFrame(iris.data, columns=iris.feature_names)

# Añadimos la columna de etiquetas (la clase de cada flor)
df['especie'] = iris.target

# Mapeamos los números (0, 1, 2) a los nombres reales de las especies
nombres_especies = {0: 'Setosa', 1: 'Versicolor', 2: 'Virginica'}
df['especie_nombre'] = df['especie'].map(nombres_especies)

print(f"Dataset cargado: {df.shape[0]} filas, {df.shape[1]} columnas")
df.head(10)

### 2.3 Inspección básica del dataset

Antes de modelar, siempre hay que responder tres preguntas:
1. ¿Hay valores nulos?
2. ¿Qué tipos de datos tenemos?
3. ¿Cuál es el rango y distribución de las variables?

Un modelo entrenado sobre datos sucios o mal entendidos dará resultados incorrectos aunque el algoritmo sea perfecto. La exploración no es opcional.

In [ ]:
print("=== Información general del dataset ===")
print(df.info())
print()

print("=== Valores nulos por columna ===")
print(df.isnull().sum())
print()

print("=== Distribución de clases ===")
print(df['especie_nombre'].value_counts())

### 2.4 Estadísticas descriptivas

El método `.describe()` nos da un resumen estadístico de cada variable numérica: media, desviación estándar, mínimo, máximo y percentiles. Es una primera pista sobre si las variables están en escalas muy diferentes entre sí (lo veremos de nuevo en la fase de preprocesamiento).

> **Fíjate en:** La longitud del pétalo tiene una media de ~3.7 cm y un rango de 1 a 6.9. La anchura del sépalo está entre 2 y 4.4. Estas escalas distintas pueden afectar a ciertos algoritmos, como veremos.

In [ ]:
df.describe().round(2)

### 2.5 Visualización exploratoria

Los números solos no siempre revelan la estructura de los datos. Un **pairplot** (gráfico de dispersión por pares) nos muestra cómo se relacionan todas las variables entre sí, coloreadas por especie.

Esto nos ayuda a responder: ¿son las especies visualmente separables? Si lo son, el problema de clasificación debería ser relativamente sencillo para un modelo de ML.

In [ ]:
sns.pairplot(df.drop(columns='especie'), hue='especie_nombre', palette='Set2')
plt.suptitle('Relaciones entre variables por especie', y=1.02, fontsize=13)
plt.show()

> **¿Qué observamos?** La especie *Setosa* (verde) se separa claramente del resto en casi todas las combinaciones de variables. *Versicolor* y *Virginica* se solapan algo más, lo que significa que el modelo tendrá más dificultad para distinguirlas. Esto ya nos da una expectativa de rendimiento antes de entrenar.

---
## Fase 3 — Preprocesamiento

### 3.1 Separar variables predictoras (X) y variable objetivo (y)

En Machine Learning siempre trabajamos con dos conjuntos:
- **X** (features): las variables que el modelo usará para aprender. Son las "entradas".
- **y** (target): la variable que queremos predecir. Es la "salida".

Esta separación es fundamental. Si accidentalmente dejamos la etiqueta dentro de X, el modelo aprenderá de ella directamente y el resultado no tendrá ningún valor real.

In [ ]:
# Variables predictoras: las cuatro medidas físicas
X = df[iris.feature_names]

# Variable objetivo: la especie (como número: 0, 1 o 2)
y = df['especie']

print(f"Tamaño de X: {X.shape}")
print(f"Tamaño de y: {y.shape}")
print(f"Clases en y: {y.unique()}")

### 3.2 División en conjunto de entrenamiento y de prueba

Nunca evaluamos un modelo con los mismos datos con los que lo entrenamos. Si lo hiciéramos, solo mediríamos si el modelo ha memorizado los datos, no si ha aprendido a generalizar.

La función `train_test_split` divide aleatoriamente el dataset:
- **80% para entrenamiento** — el modelo aprende de estos datos.
- **20% para prueba** — estos datos el modelo nunca los ha visto. Los usamos para evaluar.

El parámetro `random_state=42` fija la semilla aleatoria para que el resultado sea reproducible (siempre la misma división). Es una convención habitual en la comunidad de Data Science.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,       # 20% para test
    random_state=42,     # reproducibilidad
    stratify=y           # mantiene la proporción de clases en ambos conjuntos
)

print(f"Datos de entrenamiento: {X_train.shape[0]} muestras")
print(f"Datos de prueba:        {X_test.shape[0]} muestras")

> **¿Qué hace `stratify=y`?** Sin esta opción, podría darse el caso de que la división aleatoria dejara casi todas las muestras de una clase en el conjunto de test y ninguna en el de entrenamiento. `stratify` garantiza que cada clase esté representada en la misma proporción en ambos conjuntos.

### 3.3 Normalización de las variables (StandardScaler)

El algoritmo que vamos a usar, **K-Nearest Neighbors (KNN)**, toma decisiones basándose en la distancia entre puntos. Si una variable tiene valores en el rango 0–10 y otra en el rango 0–1000, la segunda dominará el cálculo de distancias aunque no sea más informativa.

`StandardScaler` transforma cada variable para que tenga **media 0 y desviación estándar 1**. Así todas las variables contribuyen por igual.

**Regla importante:** Ajustamos el scaler **solo con los datos de entrenamiento** (`.fit_transform(X_train)`) y luego lo aplicamos sin reajustar a los datos de test (`.transform(X_test)`). Si ajustáramos también con los datos de test, estaríamos filtrando información del futuro hacia el modelo — lo que se llama *data leakage*.

In [ ]:
scaler = StandardScaler()

# Ajustamos y transformamos los datos de entrenamiento
X_train_scaled = scaler.fit_transform(X_train)

# Solo transformamos (sin reajustar) los datos de test
X_test_scaled = scaler.transform(X_test)

# Comprobamos que la media sea ~0 y la desviación estándar ~1
print("Media por variable (train escalado):")
print(X_train_scaled.mean(axis=0).round(4))
print()
print("Desviación estándar por variable (train escalado):")
print(X_train_scaled.std(axis=0).round(4))

---
## Fase 4 — Entrenamiento del modelo

### 4.1 ¿Por qué K-Nearest Neighbors?

KNN es uno de los algoritmos de clasificación más intuitivos que existen. Su lógica es sencilla: **para clasificar un nuevo punto, busca los K puntos más cercanos en el conjunto de entrenamiento y asigna la clase más votada entre ellos.**

Es una buena elección para este proyecto porque:
- Es fácil de entender e interpretar
- Funciona bien con datasets pequeños y bien separados
- Tiene un solo hiperparámetro principal: el número de vecinos `K`

Usaremos `K=5` como punto de partida. Un valor pequeño de K hace que el modelo sea más sensible al ruido; un valor grande lo hace más estable pero menos flexible.

In [ ]:
# Creamos el modelo con K=5 vecinos
modelo = KNeighborsClassifier(n_neighbors=5)

# Entrenamos el modelo con los datos de entrenamiento escalados
modelo.fit(X_train_scaled, y_train)

print("✅ Modelo entrenado correctamente")
print(f"Algoritmo: {modelo.__class__.__name__}")
print(f"Número de vecinos (K): {modelo.n_neighbors}")

### 4.2 Generar predicciones

Una vez entrenado, el modelo puede predecir la clase de nuevas flores que nunca ha visto. Le pasamos los datos de test (los que apartamos antes) y le pedimos que clasifique cada muestra.

Las predicciones son números (0, 1, 2), que corresponden a las especies. Las convertiremos a nombres para facilitar la interpretación.

In [ ]:
# El modelo predice sobre los datos de test (que nunca ha visto)
y_pred = modelo.predict(X_test_scaled)

# Mostramos las primeras 10 predicciones frente a los valores reales
comparacion = pd.DataFrame({
    'Real':     [nombres_especies[v] for v in y_test.values[:10]],
    'Predicho': [nombres_especies[v] for v in y_pred[:10]]
})
comparacion['¿Correcto?'] = comparacion['Real'] == comparacion['Predicho']
print(comparacion.to_string(index=False))

---
## Fase 5 — Evaluación del modelo

### 5.1 Exactitud global (accuracy)

La métrica más básica es la **exactitud**: el porcentaje de predicciones correctas sobre el total. Es un buen punto de partida, aunque no siempre suficiente (un modelo que clasifica un 99% bien puede ser inútil si el 1% de errores son los casos más importantes).

En este proyecto, dado que las clases están equilibradas (50 muestras de cada especie), la exactitud es una métrica razonable.

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
print(f"Exactitud del modelo en test: {accuracy:.2%}")

### 5.2 Informe de clasificación

El informe de clasificación desglosa el rendimiento por clase, con tres métricas clave:

- **Precision:** De todas las veces que el modelo dijo "esta flor es X", ¿cuántas veces tenía razón?
- **Recall:** De todas las flores que realmente son X, ¿cuántas identificó correctamente el modelo?
- **F1-score:** Media armónica entre precision y recall. Equilibra ambas métricas.

> **Cuándo importa cada una:** Si el coste de un falso positivo es alto (diagnóstico médico erróneo), importa más la precision. Si el coste de un falso negativo es alto (no detectar un fraude), importa más el recall.

In [ ]:
print("=== Informe de Clasificación ===")
print(classification_report(
    y_test, y_pred,
    target_names=['Setosa', 'Versicolor', 'Virginica']
))

### 5.3 Matriz de confusión

La **matriz de confusión** muestra dónde se equivoca el modelo con más detalle. Cada fila representa la clase real; cada columna, la clase predicha. Los valores en la diagonal son predicciones correctas; todo lo que está fuera de la diagonal son errores.

La visualizamos como un mapa de calor para facilitar la lectura.

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 4))
sns.heatmap(
    cm,
    annot=True, fmt='d', cmap='Blues',
    xticklabels=['Setosa', 'Versicolor', 'Virginica'],
    yticklabels=['Setosa', 'Versicolor', 'Virginica']
)
plt.xlabel('Clase Predicha')
plt.ylabel('Clase Real')
plt.title('Matriz de Confusión — KNN (K=5)')
plt.tight_layout()
plt.show()

> **¿Qué buscamos en la matriz?** Idealmente, todos los valores estarían en la diagonal (predicciones perfectas). Si hay valores fuera de la diagonal, miramos qué clases se confunden entre sí. Esto conecta con lo que ya anticipamos en el pairplot: *Versicolor* y *Virginica* son las más parecidas y donde esperamos más errores.

### 5.4 Exploración del hiperparámetro K

Elegimos K=5 al principio, pero ¿es el mejor valor posible? Una práctica habitual es probar varios valores de K y ver cuál da mejor resultado en test.

Esto es una forma simplificada de **búsqueda de hiperparámetros**, uno de los pasos esenciales en cualquier proyecto real de ML.

In [ ]:
resultados_k = []

for k in range(1, 21):
    knn_k = KNeighborsClassifier(n_neighbors=k)
    knn_k.fit(X_train_scaled, y_train)
    acc = accuracy_score(y_test, knn_k.predict(X_test_scaled))
    resultados_k.append({'K': k, 'Exactitud': acc})

df_k = pd.DataFrame(resultados_k)

plt.figure(figsize=(8, 4))
plt.plot(df_k['K'], df_k['Exactitud'], marker='o', color='steelblue')
plt.axhline(y=df_k['Exactitud'].max(), color='red', linestyle='--', alpha=0.5, label='Máximo')
plt.xlabel('Número de vecinos (K)')
plt.ylabel('Exactitud en test')
plt.title('Exactitud según el valor de K')
plt.xticks(range(1, 21))
plt.legend()
plt.tight_layout()
plt.show()

mejor_k = df_k.loc[df_k['Exactitud'].idxmax()]
print(f"Mejor K: {int(mejor_k['K'])} → Exactitud: {mejor_k['Exactitud']:.2%}")

---
## Fase 6 — Reflexión y próximos pasos

### ¿Qué hemos construido?

Un sistema completo de clasificación automática que, dado un conjunto de medidas de una flor, es capaz de predecir su especie con alta exactitud. Hemos seguido el flujo de trabajo estándar de un proyecto de ML:

1. **Definir el problema** → Clasificación supervisada de tres clases
2. **Explorar los datos** → Visualizamos la separabilidad entre clases
3. **Preprocesar** → Dividimos train/test, escalamos las variables
4. **Entrenar** → KNN con K=5 (ajustado después)
5. **Evaluar** → Exactitud, informe de clasificación, matriz de confusión

### ¿Qué mejoraríamos?

Este proyecto es sencillo a propósito, pero hay muchas líneas de mejora para seguir explorando:

- **Probar otros algoritmos:** ¿Qué resultados daría un árbol de decisión o una regresión logística? Cada algoritmo tiene suposiciones distintas sobre los datos.
- **Validación cruzada:** En lugar de una única división train/test, evaluar sobre múltiples particiones (K-Fold Cross-Validation) da una estimación más robusta del rendimiento.
- **Dataset más complejo:** Iris es un dataset casi perfecto. En datos reales habrá valores nulos, desbalance de clases, variables irrelevantes, etc.
- **Despliegue:** Un modelo entrenado se puede guardar con `joblib` o `pickle` y servir como API para que otras aplicaciones lo usen.

---

### Pregunta de reflexión final

> ¿Podría usarse este mismo flujo de trabajo (explorar → preprocesar → entrenar → evaluar) para un problema completamente distinto, como predecir si un cliente va a cancelar su suscripción? ¿Qué cambiaría y qué se mantendría igual?


---
## Ejercicio propuesto (opcional / ampliación)

Adapta el proyecto para usar el **dataset Wine** en lugar del Iris. También está incluido en scikit-learn:

```python
from sklearn.datasets import load_wine
wine = load_wine()
```

Preguntas guía:
1. ¿Cuántas clases tiene el dataset Wine? ¿Cuántas variables?
2. ¿El pairplot muestra las mismas conclusiones que con Iris?
3. ¿El modelo KNN con K=5 rinde igual, mejor o peor? ¿Por qué crees que es así?
4. ¿Cambia el resultado si no escalas los datos?